### RAGAS 메트릭을 나누는 두 가지 기준

1. LLM을 쓰느냐, 안쓰느냐
  - MetricWithLLM, Non-LLM Metric

2. 싱글턴, 멀티
  - SingleTurnMetric, MultiTurnMetric

### 결과값 형태에 따른 세 가지 타입

#### 1. Discrete
- 반환값: 미리 정한 범주 중 하나
- 용도: pass/fail, good/okay/bad 분류, 빠른 커스텀 평가 적합
- 데코레이터: ```@discrete_metric```

#### 2. Numeric
- 반환값: 범위 내 정수/실수
- 용도: mean, sum, mode 같은 집계 => 통계 분석
- 데코레이터: ```@numeric_metric```

#### 3. Ranking
- 반환값: 순위 리스트
- 용도: 같은 파이프라인의 여러 출력을 서로 비교
- 데코레이터: ```@ranking_metric```

In [ ]:
# 예시
from ragas.metrics import discrete_metric, numeric_metric

# return pass or fail
@discrete_metric(name='response_quality', allowed_values=["pass", "fail"])
def my_metric(predicted: str, expected: str) -> str:
    return "pass" if predicted.lower() == expected.lower() else "fail"

# reutrn numeric
@numeric_metric(name="response_accuracy", allowed_values=(0, 1))
def acc(predicted: float, expected: float) -> float:
    return abs(predicted - expected) / max(expected, 1e-5)

acc.score(predicted=0.8, expected=1.0)

### RAGAS 설계 원칙 5가지
1. 한 메트릭엔 한 가지만
2. 메트릭의 이름은 보자마자 이해되게
3. 프롬프트는 단계별로
   - 복잡한 판단을 작은 작업으로 쪼갬 (예: 문장 → claim 분해 → 검증)
4. 점수 범위 통일

In [ ]:
# API 사용법
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings.base import embedding_factory
from ragas.metrics.collections import Faithfulness

client = AsyncOpenAI()
llm = llm_factory("gpt-4o-mini", client=client)
emb = embedding_factory(
    "openai",
    model="text-embedding-3-small",
    client=client
)

scorer = Faithfulness(llm=llm)
result = await scorer.ascore(
    user_input = "",
    response = "",
    retrieved_contexts = [""]
)
# 점수
result.value

### Metrics Maps
1. RAG
    - 검색 품질
        - Context Precision: 필요한 문서가 잘 검색 됐나?
        - Context Recall: 필요한 정보를 놓치지 않았나?
        - Entities Recall: 필요한 엔티티를 검색했나?
    - 생성 품질
        - Faithfulness: 답변이 검색 문서에 근거하나?
        - Response Relevancy: 질문에 제대로 답했나?
    - 노이즈 내성
        - Noise Sensitivity: 불필요한 문서가 있어도 답변이 흔들리지 않나?
    - 멀티모달
        - MM Faithfulness: 이미지/텍스트 근거에 충실한가?
        - MM Relevance: 멀티모달 질문에 적절하게 답했나?
2. NVIDIA가 제안하는 비교적 가벼운 평가 방식
    - Answer Accuracy: 정답이 맞는지?
    - Context Relevance: 가져온 문서가 질문과 관련이 있는지?
    - Response Groundedness: 답변이 Context에 근거하고 있나?
3. Agent/Tool
    - Topic Adherence: 대화 주제를 계속 유지하나?
    - Tool Call Accuracy: 올바른 Tool을 호출했나?
    - Tool Call F1: Tool 호출을 얼마나 정확하게 했나?
    - Agent Goal Accuracy: 최종적으로 사용자의 목적을 달성했나?
4. 비교 · SQL · 범용 · 기타
    - 문장 비교(Semantic Similarity)
    - 번역/텍스트 생성(BLEU, ROUGE, CHRF)
    - SQL Equivalence
    - LLM Judge